# Z3 的使用方法 ― 用 5 个步骤求解约束

## 本章要点

上一章我们看到，用 Z3 写约束满足问题，代码可以很短。本章按**基本流程**一步步确认 Z3 的用法。

- 用 Z3 解题的 5 个步骤
- Z3 的变量和普通 Python 数值有什么不同
- 从模型（答案）里取出数值的方法
- 例题：求解「SEND + MORE = MONEY」
- 列出所有解的方法

本章的代码请**从上往下依次执行**，后面的代码会用到前面定义的变量。

In [1]:
from z3 import *

## 用 Z3 解题的流程

用 Z3 解题的步骤，不管什么问题几乎都一样。

| 步骤 | 要做的事 | 用到的工具 |
|---|---|---|
| 1 | 创建变量 | `Int`, `Ints`, `IntVector` |
| 2 | 准备求解器 | `Solver()` |
| 3 | 添加约束 | `solver.add()` |
| 4 | 检查是否有解 | `solver.check()` |
| 5 | 取出数值 | `solver.model()` |

下面按顺序看这 5 步。

## 步骤 1：创建变量

首先，创建值还没有确定的变量。有三种创建方式。

In [2]:
x = Int("x")             # 创建一个整数变量
y, z = Ints("y z")       # 用空格分隔名字，一次创建多个
v = IntVector("v", 5)    # v[0] 〜 v[4] 共 5 个整数变量

创建出来的变量类型是 `ArithRef`。`IntVector` 是 Python 的列表，里面装的都是 `ArithRef`。

In [3]:
print(type(x), x, y, z)
print(type(v), type(v[0]), v)

<class 'z3.z3.ArithRef'> x y z
<class 'list'> <class 'z3.z3.ArithRef'> [v__0, v__1, v__2, v__3, v__4]


Z3 的变量和普通 Python 数值的行为不同。它的值还没有确定，所以参与计算也不会变成数值，会保持表达式原样。

In [4]:
print(x + 1)            # 不会计算，原样显示为表达式
print(x + 1 == 10)      # 变成条件式（BoolRef）
print(type(x + 1 == 10))

x + 1
x + 1 == 10
<class 'z3.z3.BoolRef'>


## 步骤 2：准备求解器

把约束存起来、负责寻找解的就是 `Solver`。先创建一个空的求解器。

In [5]:
solver = Solver()
print(type(solver), solver)   # [] 表示「还没有任何约束」

<class 'z3.z3.Solver'> []


## 步骤 3：添加约束

把约束传给 `solver.add()`。`==`、`<`、`>`、`+` 这些运算符可以直接写成约束表达式。

约束可以一个一个传，也可以先放进列表再一次传进去。

In [6]:
solver = Solver()

constraints = [
    x + y == 10,   # x 与 y 之和是 10
    x > 0,         # x 是正数
    y > 0,         # y 是正数
]
solver.add(*constraints)   # 展开列表传入（等同于 solver.add(a, b, c)）
print(solver)

[x + y == 10, x > 0, y > 0]


用 `print()` 打印 `solver`，会把当前的所有约束显示出来。想确认条件有没有写对时很方便。

## 步骤 4：检查是否有解

用 `Solver.check()` 检查是否存在满足约束的解。返回值有三种。

| 返回值 | 含义 |
|---|---|
| `sat` | 找到解（satisfiable） |
| `unsat` | 不存在解（unsatisfiable） |
| `unknown` | 无法判断（超时、混入了 Z3 不擅长的式子等） |

In [7]:
if solver.check() == sat:
    print("有解")
else:
    print("无解")

有解


## 步骤 5：从模型里取出数值

有解时，可以用 `solver.model()` 取出具体的值。返回的是一个叫 `ModelRef` 的对象。

In [8]:
if solver.check() == sat:
    model = solver.model()
    print("x =", model[x])
    print("y =", model[y])
else:
    print("无解")

x = 9
y = 1


`model[x]` 是 Z3 的 `IntNumRef` 类型，不能直接当作 Python 整数来算。要用 `as_long()` 转换。

In [9]:
print(type(model), type(model[x]))

print("x =", model[x].as_long())
print("x + y =", model[x].as_long() + model[y].as_long())

<class 'z3.z3.ModelRef'> <class 'z3.z3.IntNumRef'>
x = 9
x + y = 10


(z3-send-more-money)=
## 例题：SEND + MORE = MONEY

[用 OrTools 解同一个问题](ortools-send-more-money)

这是把字母换成数字的谜题（字母算术，Cryptarithm）。

> S、E、N、D、M、O、R、Y 分别表示 0〜9 中**互不相同**的数字。
>
> $$SEND + MORE = MONEY$$
>
> 请找出让这个式子成立的数字组合。

答案是 $9567 + 1085 = 10652$。我们用 Z3 实际算一遍，看看能不能得到这个答案。

解法就是本章的 5 个步骤。

1. 把 S、E、N、D、M、O、R、Y 设为整数变量
2. 添加约束：每个字母都取 0〜9，而且互不相同
3. 把单词写成算式：$SEND = 1000S + 100E + 10N + D$
4. 把 $SEND + MORE = MONEY$ 作为约束添加进去
5. 用 `check()` 和 `model()` 取出解

In [10]:
# 把 S, E, N, D, M, O, R, Y 设为整数变量
S, E, N, D, M, O, R, Y = Ints("S E N D M O R Y")
letters = [S, E, N, D, M, O, R, Y]

solver = Solver()

# 1. 每个字母都是 0〜9 之一
for letter in letters:
    solver.add(0 <= letter, letter <= 9)

# 2. 字母不同，数字也不同
solver.add(Distinct(letters))

# 3. 把单词写成算式  例：SEND = 1000*S + 100*E + 10*N + D
send = 1000*S + 100*E + 10*N + D
more = 1000*M + 100*O + 10*R + E
money = 10000*M + 1000*O + 100*N + 10*E + Y

# 4. SEND + MORE = MONEY
solver.add(send + more == money)

# 5. 最高位不能是 0
solver.add(S != 0, M != 0)

if solver.check() == sat:
    model = solver.model()
    print({str(letter): model[letter].as_long() for letter in letters})
    print(f"{model.eval(send)} + {model.eval(more)} = {model.eval(money)}")
else:
    print("无解")

{'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O': 0, 'R': 8, 'Y': 2}
9567 + 1085 = 10652


答案只有一种。代码里只用到了两个运算符。

| 运算符 | 作用 | 例子 |
|---|---|---|
| `And` | 同时满足所有条件 | `And(x > 0, y > 0)` |
| `Distinct` | 给定的变量取互不相同的值 | `Distinct(a, b, c)` |

取值也有两种方式。

- `model[x]` … 取出变量的值
- `model.eval(式)` … 取出表达式的计算结果

### 模型里有什么

`ModelRef` 把变量和值放在一起。往 `[]` 里传什么，返回的东西就不同。

| 传入 | 返回 |
|---|---|
| 变量（`ArithRef`） | 它的值（`IntNumRef`） |
| 下标（整数） | 变量的声明（`FuncDeclRef`） |

In [11]:
print(model[0], type(model[0]))   # 第 0 个变量的「声明」

decl = model[0]
print(decl(), type(decl()))       # 加上 () 就变成变量（ArithRef）

S <class 'z3.z3.FuncDeclRef'>
S <class 'z3.z3.ArithRef'>


`FuncDeclRef` 是变量的声明，加上 `()` 就得到变量本身。哪个字母排在第 0 位由 Z3 内部顺序决定，每次执行可能会变。

利用这个机制，可以一次取出模型里所有变量的值。

In [12]:
{str(decl): model[decl].as_long() for decl in model}

{'S': 9, 'O': 0, 'N': 6, 'R': 8, 'E': 5, 'M': 1, 'Y': 2, 'D': 7}

## 列出所有解

如果需要的不是「一个解」，而是「所有解」，就添加**不再输出已找到的解**这一约束，然后反复调用 `check()`。

1. 用 `check()` 检查是否有解
2. 用 `model()` 取出解
3. 添加「这个解不要再出现」的约束，回到第 1 步
4. 没有解了（`unsat`）就结束

In [13]:
abc = Ints("a b c")
solver = Solver()

for n in abc:
    solver.add(1 <= n, n <= 3)   # 都在 1〜3 之间
solver.add(Distinct(abc))        # 三个数互不相同

while solver.check() == sat:
    model = solver.model()
    print(model)
    # 禁止与刚找到的解「完全相同」的组合
    solver.add(Not(And([decl() == model[decl] for decl in model])))

[c = 2, b = 3, a = 1]
[a = 3, c = 1, b = 2]
[c = 2, b = 1, a = 3]
[c = 1, b = 3, a = 2]
[c = 3, a = 2, b = 1]
[a = 1, c = 3, b = 2]


`Not(And(c1, c2, ...))` 与 `Or(Not(c1), Not(c2), ...)` 意思相同。也就是「至少有一个不同」＝「不是刚才那个解」。

下面这一行也能做到同样的事。

```python
solver.add(Or([decl() != model[decl] for decl in model]))
```

不过这种方法每找到一个解就要调用一次 `check()`。解很多的问题会很花时间，请注意。

## 小结

用 Z3 解题的流程永远是这 5 步。

1. 创建变量（`Int`, `Ints`, `IntVector`）
2. 创建 `Solver()`
3. 用 `solver.add()` 添加约束
4. 用 `solver.check()` 确认 `sat` / `unsat` / `unknown`
5. 用 `solver.model()` 取值，再用 `as_long()` 转成 Python 整数

- Z3 的变量是「还没确定的值」，始终以表达式形式处理
- 用 `print(solver)` 检查约束，容易发现写错的地方
- 想要所有解时，添加禁止已找到解的约束并反复求解